<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/lab2_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Lab 2 Task 2：agent 评估

姓名：胡豪达

学号：523021910471

本 Task 占比本 Lab 评分的 20%，分为两个部分：

| 部分 | 内容 | 分值 |
|------|------|------|
| Part 1 | 运行给出的代码体会如何评估智能体工作质量 | 30 分 |
| Part 2 | 编写代码，动手做一做 | 70 分 |

<mark> 👉  高亮块是你需要完成或者修改的内容提示，你需要根据任务要求修改对应地方下方的代码块 </mark>，其余代码块请按顺序运行

#### 实验概述
在本实验中，你将专注于评估该工作流中的一个环节：研究（搜索）步骤。
你的任务是设计组件级评估，用来检验研究步骤所返回来源的质量。

该评估会把智能体检索到的 URL 与一份预定义的偏好域名列表（例如 arxiv.org、nature.com、nasa.gov）进行对比。这样你就可以用客观的、逐条样本的真实标准（ground truth）评估，量化系统是否从可信来源获取信息。

以下是一个案例：网络搜索结果质量较差，使人难以信任所检索到的信息。在此基础上，本实验通过将来源与预定义的偏好域名列表对比，来评估来源的可靠性。
本次评估将围绕主题 “黑洞科学近期进展”展开。目的是检验网络搜索工具是否返回来自偏好域名的来源，并量化偏好来源数与总结果数的比例。

该评估将实现为单个函数，完成客观的逐条检查。它会：
- 解析 Tavily 的输出（我们的网络搜索工具）；
- 识别哪些 URL 属于偏好域名列表；
- 计算偏好来源数与检索到的总来源数的比例；
- 返回一个布尔标志（通过/不通过）以及一段 Markdown 格式的摘要，可直接嵌入报告中。


你将学会如何：

- 编写一个函数，用于检查网络搜索 API 的搜索结果是否来自偏好来源。
- 创建一项评估，验证你的来源是否来自你设定的偏好域名。
- 在网络搜索函数中加入组件级评估。

[![Gemini-Generated-Image-4tyfwb4tyfwb4tyf.png](https://i.postimg.cc/8znmr8V4/Gemini-Generated-Image-4tyfwb4tyfwb4tyf.png)](https://postimg.cc/hXTdH5vQ)

可以参考 https://openrouter.ai/models 中的模型列表，以切换不同的 LLM

---
## 🔧 环境安装

Colab 中连接到运行时可以选用 cpu 即可

In [1]:
!pip install -q uv
!uv pip install -qqq requests openai pillow python-dotenv tavily-python "aisuite[openai]" wikipedia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.8 MB/s eta 0:00:00


In [2]:
from datetime import datetime
import json
import re
import os
from typing import Any

import pandas as pd
import wikipedia
import requests
import xml.etree.ElementTree as ET
import base64
from IPython.display import HTML, display

from aisuite import Client

In [3]:
os.environ["TAVILY_API_KEY"] = "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"
os.environ["OPENROUTER_API_KEY"] = "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"

In [4]:
client = Client(
    provider_configs={
        "openai": {
            "api_key": os.environ["OPENROUTER_API_KEY"],
            "base_url": "https://openrouter.ai/api/v1",
        }
    }
)

In [5]:
def print_html(content: Any, title: str | None = None, is_image: bool = False):
    """
    Pretty-print inside a styled card.
    - If is_image=True and content is a string: treat as image path/URL and render <img>.
    - If content is a pandas DataFrame/Series: render as an HTML table.
    - Otherwise (strings/otros): show as code/text in <pre><code>.
    """
    try:
        from html import escape as _escape
    except ImportError:
        _escape = lambda x: x

    def image_to_base64(image_path: str) -> str:
        with open(image_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode("utf-8")

    # Render content
    if is_image and isinstance(content, str):
        b64 = image_to_base64(content)
        rendered = f'<img src="data:image/png;base64,{b64}" alt="Image" style="max-width:100%; height:auto; border-radius:8px;">'
    elif isinstance(content, pd.DataFrame):
        rendered = content.to_html(classes="pretty-table", index=False, border=0, escape=False)
    elif isinstance(content, pd.Series):
        rendered = content.to_frame().to_html(classes="pretty-table", border=0, escape=False)
    elif isinstance(content, str):
        rendered = f"<pre><code>{_escape(content)}</code></pre>"
    else:
        rendered = f"<pre><code>{_escape(str(content))}</code></pre>"

    css = """
    <style>
    .pretty-card{
      font-family: ui-sans-serif, system-ui;
      border: 2px solid transparent;
      border-radius: 14px;
      padding: 14px 16px;
      margin: 10px 0;
      background: linear-gradient(#fff, #fff) padding-box,
                  linear-gradient(135deg, #3b82f6, #9333ea) border-box;
      color: #111;
      box-shadow: 0 4px 12px rgba(0,0,0,.08);
    }
    .pretty-title{
      font-weight:700;
      margin-bottom:8px;
      font-size:14px;
      color:#111;
    }
    /* 🔒 Solo afecta lo DENTRO de la tarjeta */
    .pretty-card pre,
    .pretty-card code {
      background: #f3f4f6;
      color: #111;
      padding: 8px;
      border-radius: 8px;
      display: block;
      overflow-x: auto;
      font-size: 13px;
      white-space: pre-wrap;
    }
    .pretty-card img { max-width: 100%; height: auto; border-radius: 8px; }
    .pretty-card table.pretty-table {
      border-collapse: collapse;
      width: 100%;
      font-size: 13px;
      color: #111;
    }
    .pretty-card table.pretty-table th,
    .pretty-card table.pretty-table td {
      border: 1px solid #e5e7eb;
      padding: 6px 8px;
      text-align: left;
    }
    .pretty-card table.pretty-table th { background: #f9fafb; font-weight: 600; }
    </style>
    """

    title_html = f'<div class="pretty-title">{title}</div>' if title else ""
    card = f'<div class="pretty-card">{title_html}{rendered}</div>'
    display(HTML(css + card))

---
## 🚀 1. 网络搜索步骤

首先我们提供了 3 个 tool，用于从 3 处不同的信息来源搜索信息：
- arxiv
- tavily
- wikipedia

In [6]:
def arxiv_search_tool(query: str, max_results: int = 5) -> list[dict]:
    """
    Searches arXiv for research papers matching the given query.
    """
    url = f"https://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return [{"error": str(e)}]

    try:
        root = ET.fromstring(response.content)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}

        results = []
        for entry in root.findall('atom:entry', ns):
            title = entry.find('atom:title', ns).text.strip()
            authors = [author.find('atom:name', ns).text for author in entry.findall('atom:author', ns)]
            published = entry.find('atom:published', ns).text[:10]
            url_abstract = entry.find('atom:id', ns).text
            summary = entry.find('atom:summary', ns).text.strip()

            link_pdf = None
            for link in entry.findall('atom:link', ns):
                if link.attrib.get('title') == 'pdf':
                    link_pdf = link.attrib.get('href')
                    break

            results.append({
                "title": title,
                "authors": authors,
                "published": published,
                "url": url_abstract,
                "summary": summary,
                "link_pdf": link_pdf
            })

        return results
    except Exception as e:
        return [{"error": f"Parsing failed: {str(e)}"}]

In [7]:
def tavily_search_tool(query: str, max_results: int = 5, include_images: bool = False) -> list[dict]:
    """
    使用 Tavily API 进行搜索.

    Args:
        query (str): 搜索查询字符串。
        max_results (int): 返回结果条数（默认 5）。
        include_images (bool): 是否包含图片类结果。

    Returns:
        list[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    params = {}
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")
    params['api_key'] = api_key

    #client = TavilyClient(api_key)

    api_base_url = os.getenv("DLAI_TAVILY_BASE_URL")
    if api_base_url:
        params['api_base_url'] = api_base_url

    client = TavilyClient(api_key=api_key, api_base_url=api_base_url)

    try:
        response = client.search(
            query=query,
            max_results=max_results,
            include_images=include_images
        )

        results = []
        for r in response.get("results", []):
            results.append({
                "title": r.get("title", ""),
                "content": r.get("content", ""),
                "url": r.get("url", "")
            })

        if include_images:
            for img_url in response.get("images", []):
                results.append({"image_url": img_url})

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [8]:
def wikipedia_search_tool(query: str, sentences: int = 5) -> list[dict]:
    """
    根据给定查询在维基百科中检索并返回摘要。

    Args:
        query (str): 用于维基百科的搜索查询。
        sentences (int): 摘要中包含的句子数量。

    Returns:
        list[dict]: A list with a single dictionary containing title, summary, and URL.
    """
    try:
        page_title = wikipedia.search(query)[0]
        page = wikipedia.page(page_title)
        summary = wikipedia.summary(page_title, sentences=sentences)

        return [{
            "title": page.title,
            "summary": summary,
            "url": page.url
        }]
    except Exception as e:
        return [{"error": str(e)}]

在这里，我们把网络搜索能力拆成独立函数 find_references。find_references 的作用是从 Arxiv、Tavily、Wikipedia 等工具中收集外部信息。由于这些结果的质量会直接影响计分实验的输出，这一阶段很适合应用评估方法——例如检查返回的 URL 是否来自你列出的偏好域名。


In [9]:
def find_references(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """Perform a research task using external tools (arxiv, tavily, wikipedia)."""

    prompt = f"""
    你是一个研究型函数，可以使用：
    - arxiv_tool：学术论文
    - tavily_tool：通用网页搜索（若被要求则返回 JSON）
    - wikipedia_tool：百科式摘要

    任务：
    {task}

    今天是 {datetime.now().strftime('%Y-%m-%d')}。
    """.strip()

    messages = [{"role": "user", "content": prompt}]
    tools = [
        arxiv_search_tool,
        tavily_search_tool,
        wikipedia_search_tool,
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            max_turns=5,
        )
        content = response.choices[0].message.content
        return (content, messages) if return_messages else content
    except Exception as e:
        return f"[Model Error: {e}]"

尝试使用 find_references 来获取参考

In [20]:
research_task = "查找 5 篇与机器人世界模型近期进展相关的较新论文。"
research_result = find_references(research_task)

print_html(
    research_result,
    title="研究函数输出"
)

---
## 🚀 2. 评估步骤 – 偏好域名

网络搜索检索到的来源并非同样可靠，在本实验中，我们关注如何设计组件级评估，以检查返回的域名是否属于预定义的偏好域名列表。

这是客观评估的一个例子：对每条样本都有明确的真实标准（ground truth）。回顾课堂内容，评估有两个维度；在本任务中，我们处于左上象限——即对每个样本应用、且真实标准明确定义的客观评估。


[![jie-ping2026-04-10-17-50-02.png](https://i.postimg.cc/mk5gyS2x/jie-ping2026-04-10-17-50-02.png)](https://postimg.cc/GTvRcGyz)

#### 为什么要做组件级评估？

- 若问题出在网络搜索（通常是计分实验工作流的第一步），每次都重跑整条流水线（搜索 → 初稿 → 反思）既费资源又噪声大。
- 搜索质量的细微提升，可能被后续环节引入的随机性掩盖。
- 单独评估网络搜索，能更清楚地判断该组件是否在变好。
- 当多个团队负责系统不同部分时，组件级评估也很高效：各团队可用明确指标优化自己的模块，而不必每次跑或等待完整的端到端测试。

#### 如何评估？

这里的评估是客观的，因此可以用代码实现。它对每条样本有对应的真实标准——针对这道黑洞查询的偏好来源列表。构建该评估时，你需要：

- 提取 Tavily 返回的 URL。
- 与预定义的偏好域名列表比对（例如 arxiv.org、nature.com、nasa.gov）。
- 计算偏好结果数与总结果数的比例。
- 返回通过/不通过标志，以及一段 Markdown 格式的摘要。

这样得到一个可重复、成本低的指标，用于判断研究组件——也就是计分实验中仅这一步——是否从可信来源拉取信息。

#### 为什么这是客观评估：

从 Tavily 获取的每个 URL 都会与预定义的偏好域名列表（TOP_DOMAINS）比对：
- 若域名匹配 → 偏好
- 否则 → 非偏好

据此，根据偏好来源占比是否超过给定阈值，可以得到明确的通过/不通过信号。由于对每条样本而言，真实标准（偏好 vs. 非偏好）是明确定义的，因此该评估既是客观的，也是可重复的。

In [17]:
TOP_DOMAINS = {
    # General reference / institutions / publishers
    "wikipedia.org", "nature.com", "science.org", "sciencemag.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov", "europa.eu", "sjtu.edu.cn"

    # CS/AI venues & indexes
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",

    # Other reputable outlets
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",

    # Extra domains (case-specific additions)
    "pbs.org", "nova.edu", "nvcc.edu", "cccco.edu",

    # Well known programming sites
    "codecademy.com", "datacamp.com"
}

def evaluate_tavily_results(TOP_DOMAINS, raw: str, min_ratio=0.4):
    """
    评估纯文本形式的研究结果是否主要来自偏好域名。

    Args:
        TOP_DOMAINS (set[str]): 偏好域名集合（例如 'arxiv.org'、'nature.com'）。
        raw (str): 包含 URL 的纯文本或 Markdown。
        min_ratio (float): 通过所需的最低「偏好占比」（例如 0.4 表示 40%）。

    Returns:
        tuple[bool, str]: (flag, markdown_report)
            flag -> True 表示通过（PASS），False 表示未通过（FAIL）
            markdown_report ->  Markdown 格式的评估摘要
    """

    # Extract URLs from the text
    url_pattern = re.compile(r'https?://[^\s\]\)>\}]+', flags=re.IGNORECASE)
    urls = url_pattern.findall(raw)

    if not urls:
        return False, """### Evaluation — Tavily Preferred Domains
        No URLs detected in the provided text.
        Please include links in your research results.
        """

    # Count preferred vs total
    total = len(urls)
    preferred_count = 0
    details = []

    for url in urls:
        domain = url.split("/")[2]
        preferred = any(td in domain for td in TOP_DOMAINS)
        if preferred:
            preferred_count += 1
        details.append(f"- {url} → {'✅ PREFERRED' if preferred else '❌ NOT PREFERRED'}")

    ratio = preferred_count / total if total > 0 else 0.0
    flag = ratio >= min_ratio

    # Markdown report
    report = f"""
    ### 评估 —— Tavily 偏好域名
    - 结果总数：{total}
    - 偏好来源数：{preferred_count}
    - 占比：{ratio:.2%}
    - 阈值：{min_ratio:.0%}
    - 状态：{"✅ 通过" if flag else "❌ 未通过"}
    **明细：**

    {chr(10).join(details)}
    """
    return flag, report

试着运行下面的代码块，展示示例偏好域名、研究结果，以及评估摘要（通过/不通过及详细说明）。

In [21]:
print_html(json.dumps(list(TOP_DOMAINS)[:4], indent=2), title="可信域名示例")

print_html("<h3>研究结果</h3>" + research_result, title="研究结果")

flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
print_html("<pre>" + report + "</pre>", title="<h3>评估摘要</h3>")

### 👉  请在你成功运行上面所有代码块后，用一两句话简要写出你的心得体会

（例如：为什么要评估网络搜索的结果？为什么域名是否在白名单里算客观、可重复；和用LLM打分的相关性/有用性有什么不同和缺陷？除了用 TOP DOMAINS 外还有什么评估方式？ 等等 ... ）

<mark>网络搜索检索到的来源并非总是可靠，及时评估网络搜索的结果可以避免资源浪费，例如上述搜索的五篇均未通过评估。判断域名是否在白名单里一方面简单快捷，另一方面白名单内的域名通常经过人工检验，相对可靠。</mark>

## 3. 动手做一做

<mark> 👉 你需要自定义研究主题及其评估依据（在这里为设置对应的偏好域名列表）：</mark>

- 主题：换一个你所在专业的前沿研究的主题。
- 偏好域名：编辑或扩充 TOP_DOMAINS 列表为你所在专业的研究领域可信的网站。
- 评估比例：调整 min_ratio（例如 0.4 表示至少 40% 为偏好来源）。

<mark> 尝试修改下方代码块的字段 </mark>

In [26]:
topic = "Single Pilot Operation"
min_ratio = 0.4
run_reflection = False
TOP_DOMAINS = {
    "wikipedia.org", "nature.com", "science.org", "arxiv.org",
    "nasa.gov", "mit.edu", "stanford.edu", "harvard.edu",
    "sjtu.edu.cn"
}

<mark> 运行评估流程！ </mark>

In [28]:
import json
print_html(
    json.dumps(sorted(list(TOP_DOMAINS)), indent=2),
    title="<h3>可信域名示例</h3>"
)

# 1) Research
research_task = f"查找 10 篇与「{topic}」相关的重要论文及可靠综述."
research_output = find_references(research_task)
print_html(research_output, title=f"<h3>关于「{topic}」的研究结果</h3>")

# 2) Evaluate sources (preferred domains ratio)
flag, eval_md = evaluate_tavily_results(TOP_DOMAINS, research_output, min_ratio=min_ratio)
print_html("<pre>" + eval_md + "</pre>", title="<h3>评估摘要</h3>")

---
## 4. 要点回顾

- 你刚刚看到了如何评估单个组件的性能：find_references 研究步骤。
- 你的组件级评估检验了检索到的 URL 是否落在预定义的偏好域名列表中。
- 这是客观评估的一个例子：对每条样本都有明确的真实标准（ground truth）。
- 若要构建评估集，可以设计约 10 条提示，覆盖不同主题（天文学、机器人、金融等），并为每条定义对应的偏好域名。
- 检索结果中与偏好域名列表匹配的比例，是一个有用的指标，可用于指导改进——例如调整提示词或工具参数。
- 与对完整文章做反思与重写式评估相比，这种做法更简单、成本更低，因为你只关注网络搜索这一组件。

恭喜！
- 你设计了一项组件级评估，使研究型智能体更可靠。
- 通过直接检验来源质量，你引入了一道客观、可重复且成本低的保障。

组件级评估让你可以单独测试 AI 系统的各个部分，而不必承担评估整条流水线的开销。